# StockLens — Universal LSTM (S&P 500)

Entrena **un único modelo** con datos de todos los tickers del S&P 500.
El modelo aprende patrones universales del mercado y funciona para
**cualquier acción**, incluso las que no vio durante el entrenamiento.

**Tiempo estimado:** ~2-3 horas con GPU T4 de Colab gratuito

**Output:** un único archivo `lstm_universal.onnx` + `lstm_universal.pkl`
que subes a `stocklens-python/app/models/`

---
**Instrucciones:**
1. Menú → Entorno de ejecución → Cambiar tipo → GPU T4
2. Ejecuta todas las celdas en orden
3. Al final descarga `stocklens_universal.zip` y sube a Railway

In [ ]:
# ── Celda 1: Instalación ──────────────────────────────────────────────────────
!pip install -q yfinance pandas-ta shap onnx onnxruntime torch onnxscript
print('✅ Dependencias instaladas')

In [ ]:
# ── Celda 2: Configuración ────────────────────────────────────────────────────
import os
os.makedirs('models', exist_ok=True)

# S&P 500 completo + tickers extra de StockLens
SP500_TICKERS = [
    # Mega cap tech
    'AAPL','MSFT','NVDA','GOOGL','AMZN','META','TSLA','AVGO','ORCL','AMD',
    # Financials
    'JPM','BAC','GS','MS','WFC','BLK','C','AXP','SCHW','COF','V','MA','PYPL',
    # Healthcare / Biotech
    'UNH','JNJ','LLY','ABBV','MRK','TMO','ABT','AMGN','GILD','REGN',
    'MRNA','BIIB','VRTX','BMRN','PFE','AZN','NVO',
    # Energy
    'XOM','CVX','COP','SLB','OXY','MPC','PSX','VLO','EOG','DVN',
    # Consumer
    'HD','MCD','NKE','SBUX','TGT','COST','WMT','LOW','TJX','AMZN',
    # Industrials
    'CAT','DE','GE','HON','RTX','LMT','BA','NOC','UPS','FDX',
    # Semis
    'INTC','QCOM','TXN','MRVL','AMAT','LRCX','KLAC','SMCI','ARM',
    # Growth / Tech
    'CRM','NOW','SNOW','PLTR','COIN','SHOP','UBER','LYFT','DASH','ABNB',
    'ZM','DDOG','NET','MDB','OKTA','CRWD','PANW','ZS','FTNT',
    # Media / Telecom
    'NFLX','DIS','CMCSA','T','VZ','TMUS','SPOT','RBLX',
    # ETFs (representan el mercado general)
    'SPY','QQQ','IWM','XLK','XLF','XLV','XLE','ARKK',
    # Small/mid cap high-beta
    'IONQ','RKLB','ACHR','JOBY','ASTS','SOUN','MSTR','HOOD','SOFI','AFRM','UPST',
    # Más S&P 500
    'ADBE','CSCO','IBM','INTU','ANET','KLAC','MCHP','MPWR','ON','SWKS',
    'ACN','SAP','CDNS','SNPS','ANSS','PTC','EPAM','CTSH','INFY','WIT',
    'BRK-B','BX','KKR','APO','CG','ARES','TPG',
    'SPGI','MCO','ICE','CME','CBOE','NDAQ',
    'PG','KO','PEP','PM','MO','CL','EL','CHD','CLX',
    'ABBV','BMY','JAZZ','ALNY','SRPT','BEAM','EDIT',
    'NEE','DUK','SO','D','PCG','EXC','AEP','SRE',
    'AMT','PLD','EQIX','CCI','WELL','DLR','SPG','O',
    'LIN','APD','SHW','ECL','PPG','EMN','DD','DOW',
    'CVS','CI','HUM','MOH','CNC','ELV','HCA',
    'DHR','STE','EW','ISRG','MDT','BSX','ZBH','SYK','BDX',
]

# Deduplicate preserving order
seen = set(); TICKERS = []
for t in SP500_TICKERS:
    if t not in seen: seen.add(t); TICKERS.append(t)

# Hyperparámetros del modelo universal
TIMESTEPS    = 30      
HIDDEN_SIZE  = 128     
NUM_LAYERS   = 3       
DROPOUT      = 0.2     # Reducido para que aprenda más rápido al principio
EPOCHS       = 150     # Aumentado para darle tiempo a salir de la media
LR           = 0.001   # Aumentado para pasos más agresivos
BATCH_SIZE   = 128     # Reducido para meter "ruido" sano en los gradientes
TRAIN_SPLIT  = 0.85
PERIOD       = '4y'    
MC_SAMPLES   = 50
MAX_TICKERS  = len(TICKERS)

print(f'✅ {len(TICKERS)} tickers configurados')
print(f'   Arquitectura: LSTM({HIDDEN_SIZE}x{NUM_LAYERS}) | dropout={DROPOUT}')
print(f'   Batch={BATCH_SIZE} | Epochs={EPOCHS} | Period={PERIOD}')

In [ ]:
# ── Celda 3: Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_ta as ta
import pickle, warnings, time
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── Celda 4: Feature engineering universal ───────────────────────────────────
# Features normalizadas que funcionan para CUALQUIER acción
# La clave: usamos features RELATIVAS (retornos, distancias %) en vez de
# precios absolutos, para que el modelo sea agnóstico al nivel de precio

FEAT_COLS = [
    # Momentum
    'ret_1d', 'ret_3d', 'ret_5d', 'ret_10d', 'ret_20d',
    # Trend
    'dist_sma10', 'dist_sma20', 'dist_sma50',
    # Oscillators
    'rsi_14', 'macd_norm', 'macd_signal_norm', 'macd_hist_norm',
    # Volatility
    'bb_pct_b', 'bb_width', 'atr_norm', 'vol_20d',
    # Volume
    'vol_ratio', 'obv_norm',
    # Candle
    'high_low_pct', 'close_open_pct',
    # Stochastic
    'stoch_k', 'stoch_d',
]
N_FEATURES = len(FEAT_COLS)

def compute_features(df: pd.DataFrame) -> pd.DataFrame | None:
    """Calcula features normalizadas para un ticker."""
    if len(df) < 60:
        return None

    closes = df['Close']; highs = df['High']
    lows   = df['Low'];   vols  = df['Volume']

    out = pd.DataFrame(index=df.index)

    # Returns
    for n in [1,3,5,10,20]:
        out[f'ret_{n}d'] = closes.pct_change(n).clip(-0.2, 0.2)

    # Distance from MAs (relative, ticker-agnostic)
    for length, col in [(10,'sma10'),(20,'sma20'),(50,'sma50')]:
        sma = closes.rolling(length).mean()
        out[f'dist_{col}'] = ((closes - sma) / (sma + 1e-9)).clip(-0.3, 0.3)

    # RSI (already 0-100, normalize to 0-1)
    rsi_s = ta.rsi(closes, length=14)
    out['rsi_14'] = (rsi_s / 100.0).fillna(0.5) if rsi_s is not None else 0.5

    # MACD (normalize by close price)
    macd_df = ta.macd(closes, fast=12, slow=26, signal=9)
    if macd_df is not None:
        mcol = next((c for c in macd_df.columns if c.startswith('MACD_12')), None)
        scol = next((c for c in macd_df.columns if c.startswith('MACDs_')), None)
        hcol = next((c for c in macd_df.columns if c.startswith('MACDh_')), None)
        if mcol: out['macd_norm']        = (macd_df[mcol] / (closes + 1e-9)).clip(-0.05, 0.05)
        if scol: out['macd_signal_norm'] = (macd_df[scol] / (closes + 1e-9)).clip(-0.05, 0.05)
        if hcol: out['macd_hist_norm']   = (macd_df[hcol] / (closes + 1e-9)).clip(-0.05, 0.05)

    # Bollinger
    bb = ta.bbands(closes, length=20, std=2)
    if bb is not None:
        bbu = next((c for c in bb.columns if c.startswith('BBU_')), None)
        bbl = next((c for c in bb.columns if c.startswith('BBL_')), None)
        bbm = next((c for c in bb.columns if c.startswith('BBM_')), None)
        if bbu and bbl and bbm:
            bw = bb[bbu] - bb[bbl]
            out['bb_pct_b'] = ((closes - bb[bbl]) / (bw + 1e-9)).clip(-0.5, 1.5)
            out['bb_width'] = (bw / (bb[bbm] + 1e-9)).clip(0, 0.2)

    # ATR (normalize by close)
    atr = ta.atr(highs, lows, closes, length=14)
    if atr is not None:
        out['atr_norm'] = (atr / (closes + 1e-9)).clip(0, 0.1)

    # Volatility
    out['vol_20d'] = closes.pct_change().rolling(20).std().clip(0, 0.1)

    # Volume
    vol_sma = vols.rolling(20).mean()
    out['vol_ratio'] = (vols / (vol_sma + 1e-9)).clip(0, 5)

    # OBV (normalized, detrended)
    obv = ta.obv(closes, vols)
    if obv is not None:
        obv_norm = (obv - obv.rolling(20).mean()) / (obv.rolling(20).std() + 1e-9)
        out['obv_norm'] = obv_norm.clip(-3, 3)

    # Candle features
    out['high_low_pct']   = ((highs - lows) / (closes + 1e-9)).clip(0, 0.1)
    out['close_open_pct'] = ((closes - df['Open']) / (df['Open'] + 1e-9)).clip(-0.1, 0.1)

    # Stochastic
    stoch = ta.stoch(highs, lows, closes)
    if stoch is not None:
        k_col = next((c for c in stoch.columns if c.startswith('STOCHk_')), None)
        d_col = next((c for c in stoch.columns if c.startswith('STOCHd_')), None)
        if k_col: out['stoch_k'] = (stoch[k_col] / 100.0).fillna(0.5)
        if d_col: out['stoch_d'] = (stoch[d_col] / 100.0).fillna(0.5)

    # Keep only FEAT_COLS, fill missing with 0
    for col in FEAT_COLS:
        if col not in out.columns:
            out[col] = 0.0

    out = out[FEAT_COLS]
    out['target'] = closes.pct_change(1).shift(-1).clip(-0.15, 0.15)  # next day return
    out['close']  = closes

    return out.dropna()

print(f'✅ Feature engineering definido — {N_FEATURES} features universales')
print(f'   Features: {FEAT_COLS}')

In [ ]:
# ── Celda 5: Descargar datos de todos los tickers ────────────────────────────
print(f'Descargando {len(TICKERS)} tickers (puede tardar ~5-10 min)...')
t0 = time.time()

# yfinance batch download es mucho más rápido que uno a uno
all_data = yf.download(
    TICKERS, period=PERIOD, interval='1d',
    auto_adjust=True, progress=True,
    group_by='ticker', threads=True,
)

print(f'✅ Descarga completada en {time.time()-t0:.0f}s')

In [ ]:
# ── Celda 6: Construir dataset universal ─────────────────────────────────────
print('Construyendo dataset universal...')
t0 = time.time()

all_X, all_y = [], []
ok_tickers, failed_tickers = [], []

for ticker in TICKERS:
    try:
        # Extract single ticker from batch download
        if isinstance(all_data.columns, pd.MultiIndex):
            if ticker not in all_data.columns.get_level_values(0):
                continue
            df = all_data[ticker].dropna(subset=['Close'])
        else:
            df = all_data.dropna(subset=['Close'])

        if len(df) < 100:
            continue

        feat_df = compute_features(df)
        if feat_df is None or len(feat_df) < TIMESTEPS + 10:
            continue

        X_raw = feat_df[FEAT_COLS].values.astype(np.float32)
        y_raw = feat_df['target'].values.astype(np.float32)

        # Build sequences
        for i in range(TIMESTEPS, len(X_raw) - 1):
            all_X.append(X_raw[i-TIMESTEPS:i])
            all_y.append(y_raw[i])

        ok_tickers.append(ticker)

    except Exception as e:
        failed_tickers.append(ticker)

all_X = np.array(all_X, dtype=np.float32)
all_y = np.array(all_y, dtype=np.float32)

print(f'✅ Dataset: {all_X.shape[0]:,} secuencias de {TIMESTEPS} días × {N_FEATURES} features')
print(f'   Tickers OK: {len(ok_tickers)} | Fallidos: {len(failed_tickers)}')
print(f'   Tiempo: {time.time()-t0:.0f}s')
print(f'   Memoria: {all_X.nbytes / 1e6:.0f} MB')

In [ ]:
# ── Celda 7: Scaler y split ───────────────────────────────────────────────────
# Shuffle antes de split (mezclamos tickers para evitar data leakage)
idx = np.random.permutation(len(all_X))
all_X = all_X[idx]
all_y = all_y[idx]

split = int(len(all_X) * TRAIN_SPLIT)
X_tr, y_tr = all_X[:split], all_y[:split]
X_te, y_te = all_X[split:], all_y[split:]

# Scale target (returns) — features ya están normalizadas por construcción
scaler_y = StandardScaler()
y_tr_s = scaler_y.fit_transform(y_tr.reshape(-1,1)).ravel().astype(np.float32)
y_te_s = scaler_y.transform(y_te.reshape(-1,1)).ravel().astype(np.float32)

print(f'✅ Train: {len(X_tr):,} | Test: {len(X_te):,}')
print(f'   y_train mean: {y_tr.mean():.5f} | std: {y_tr.std():.5f}')
print(f'   Rango retornos: [{y_tr.min()*100:.1f}%, {y_tr.max()*100:.1f}%]')

In [ ]:
# ── Celda 8: Modelo LSTM universal ───────────────────────────────────────────
class UniversalLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=HIDDEN_SIZE,
                 num_layers=NUM_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 16),
            nn.GELU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


model = UniversalLSTM(input_size=N_FEATURES).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'✅ Modelo Universal: {total_params:,} parámetros')
print(f'   Arquitectura: LSTM({N_FEATURES}→{HIDDEN_SIZE}×{NUM_LAYERS}) → FC(64→16→1)')

In [ ]:
# ── Celda 9: Entrenamiento ────────────────────────────────────────────────────
tr_ds = torch.utils.data.TensorDataset(
    torch.tensor(X_tr), torch.tensor(y_tr_s))
tr_dl = torch.utils.data.DataLoader(
    tr_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True)

X_te_t = torch.tensor(X_te).to(device)
y_te_t = torch.tensor(y_te_s).to(device)

opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=0.001,
    steps_per_epoch=len(tr_dl),
    epochs=EPOCHS,
    pct_start=0.3,
)
loss_fn = nn.HuberLoss(delta=0.5)

best_val_loss = float('inf')
best_state    = None
history       = []
t0 = time.time()

print('Iniciando entrenamiento...')
print(f'  {len(tr_ds):,} muestras | batch={BATCH_SIZE} | {len(tr_dl)} batches/epoch')

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    for xb, yb in tr_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        epoch_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(X_te_t), y_te_t).item()
    sched.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    avg_train = epoch_loss / len(tr_dl)
    history.append({'epoch': epoch+1, 'train': avg_train, 'val': val_loss})

    if (epoch+1) % 10 == 0:
        elapsed = time.time() - t0
        eta     = elapsed / (epoch+1) * (EPOCHS - epoch - 1)
        print(f'  Epoch {epoch+1:3d}/{EPOCHS} — train: {avg_train:.5f} | val: {val_loss:.5f} | ETA: {eta/60:.1f} min')

model.load_state_dict(best_state)
model.eval()
print(f'\n✅ Entrenamiento completado en {(time.time()-t0)/60:.1f} min')
print(f'   Mejor val_loss: {best_val_loss:.6f}')

In [ ]:
# ── Celda 10: Evaluación ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

# Curvas de pérdida
hist_df = pd.DataFrame(history)
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(hist_df['epoch'], hist_df['train'], label='Train')
plt.plot(hist_df['epoch'], hist_df['val'],   label='Val')
plt.xlabel('Epoch'); plt.ylabel('Huber Loss')
plt.title('Training curves'); plt.legend(); plt.grid(alpha=0.3)

# Dirección prediction accuracy
with torch.no_grad():
    preds_norm = model(X_te_t[:2000]).cpu().numpy()
preds_ret  = scaler_y.inverse_transform(preds_norm.reshape(-1,1)).ravel()
true_ret   = y_te[:2000]

direction_acc = np.mean(np.sign(preds_ret) == np.sign(true_ret)) * 100
print(f'Direction accuracy: {direction_acc:.1f}%  (baseline: 50.0%)')

mask = np.abs(true_ret) > 0.002
if mask.any():
    mape = np.mean(np.abs((true_ret[mask] - preds_ret[mask]) / (true_ret[mask] + 1e-9))) * 100
    print(f'MAPE sobre retornos: {mape:.1f}%')

plt.subplot(1,2,2)
plt.scatter(true_ret[:500]*100, preds_ret[:500]*100, alpha=0.3, s=5)
plt.axline((0,0), slope=1, color='red', lw=1)
plt.xlabel('True return (%)'); plt.ylabel('Predicted return (%)')
plt.title(f'Pred vs True | Dir.Acc={direction_acc:.1f}%')
plt.xlim(-10,10); plt.ylim(-10,10); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Celda 11: Feature importance con Permutation Importance ──────────────────
# Más rápido que SHAP para dataset grande, igualmente válido
print('Calculando feature importance (permutation)...')

X_val_t = torch.tensor(X_te[:1000]).to(device)
y_val   = y_te[:1000]

model.eval()
with torch.no_grad():
    base_preds  = scaler_y.inverse_transform(model(X_val_t).cpu().numpy().reshape(-1,1)).ravel()
base_dir_acc = np.mean(np.sign(base_preds) == np.sign(y_val))

importances = {}
for fi, feat in enumerate(FEAT_COLS):
    X_perm = X_te[:1000].copy()
    np.random.shuffle(X_perm[:, :, fi])  # shuffle feature fi across all timesteps
    with torch.no_grad():
        perm_preds = scaler_y.inverse_transform(
            model(torch.tensor(X_perm).to(device)).cpu().numpy().reshape(-1,1)
        ).ravel()
    perm_acc = np.mean(np.sign(perm_preds) == np.sign(y_val))
    importances[feat] = max(0.0, float(base_dir_acc - perm_acc))

total = sum(importances.values()) + 1e-9
feat_importance = dict(sorted(
    {k: round(v/total, 4) for k,v in importances.items()}.items(),
    key=lambda x: x[1], reverse=True
))

print('\nTop 10 features:')
for feat, imp in list(feat_importance.items())[:10]:
    bar = '█' * int(imp * 200)
    print(f'  {feat:25s} {imp:.4f} {bar}')

In [ ]:
# ── Celda 12: Export a ONNX ───────────────────────────────────────────────────
model.eval()
dummy = torch.zeros(1, TIMESTEPS, N_FEATURES).to(device)

onnx_path = 'models/lstm_universal.onnx'
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
    opset_version=17,
    do_constant_folding=True,
)

onnx_size = Path(onnx_path).stat().st_size / 1e6
print(f'✅ ONNX exportado: {onnx_path} ({onnx_size:.1f} MB)')

# Save metadata
meta = {
    'feat_cols':       FEAT_COLS,
    'n_features':      N_FEATURES,
    'timesteps':       TIMESTEPS,
    'hidden_size':     HIDDEN_SIZE,
    'num_layers':      NUM_LAYERS,
    'dropout':         DROPOUT,
    'feat_importance': feat_importance,
    'direction_acc':   round(float(base_dir_acc) * 100, 2),
    'train_samples':   len(X_tr),
    'test_samples':    len(X_te),
    'ok_tickers':      ok_tickers,
    'n_tickers':       len(ok_tickers),
    'period':          PERIOD,
    'trained_at':      pd.Timestamp.now().isoformat(),
}

pkl_path = 'models/lstm_universal.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump({'scaler_y': scaler_y, 'meta': meta}, f)

print(f'✅ Metadata guardada: {pkl_path}')
print(f'   Direction accuracy: {meta["direction_acc"]}%')
print(f'   Tickers entrenados: {meta["n_tickers"]}')

In [ ]:
# ── Celda 13: Test de inferencia ─────────────────────────────────────────────
import onnxruntime as ort

sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])

print('Test de inferencia ONNX en tickers seleccionados:')
test_tickers = ['AAPL', 'NVDA', 'JPM', 'BIIB', 'UBER'][:5]

for ticker in test_tickers:
    try:
        if isinstance(all_data.columns, pd.MultiIndex) and ticker in all_data.columns.get_level_values(0):
            df_t = all_data[ticker].dropna(subset=['Close'])
        else:
            continue

        feat_df = compute_features(df_t)
        if feat_df is None or len(feat_df) < TIMESTEPS:
            continue

        X_win = feat_df[FEAT_COLS].values[-TIMESTEPS:].astype(np.float32).reshape(1, TIMESTEPS, N_FEATURES)
        pred_norm = sess.run(None, {'input': X_win})[0][0]
        pred_ret  = float(scaler_y.inverse_transform([[pred_norm]])[0][0])
        last_close = float(df_t['Close'].dropna().iloc[-1])

        print(f'  {ticker:8s}: ret_1d={pred_ret*100:+.3f}%  |  close=${last_close:.2f}')
    except Exception as e:
        print(f'  {ticker}: Error — {e}')

print('\n✅ Inferencia ONNX verificada')

In [ ]:
# ── Celda 14: Descargar modelos ───────────────────────────────────────────────
import shutil
from google.colab import files

shutil.make_archive('stocklens_universal', 'zip', 'models')
files.download('stocklens_universal.zip')

print('✅ Descarga iniciada')
print()
print('📋 Siguientes pasos:')
print('   1. Descomprime stocklens_universal.zip')
print('   2. Copia lstm_universal.onnx y lstm_universal.pkl')
print('      a stocklens-python/app/models/')
print('   3. Haz git add + commit + push')
print('   4. Railway redesplegará automáticamente')
print()
print(f'   Modelo entrenado con {meta["n_tickers"]} tickers')
print(f'   Direction accuracy: {meta["direction_acc"]}%')